# The "Hamilton Heave" PLR Power Showcase

Showcase: one resource tree for a whole facility, drawn live in the 3D visualizer

Devices (all simulated, three tiers of nine):
- Hamilton STAR, STARlet and STARplus
- Hamilton Prep

Resources:
- STAR decks: a tip carrier (1000 uL, 1000 uL filter) and two plate carriers of cor_96_wellplate_360uL_Fb
- Prep decks: cor_96_wellplate_360uL_Fb and hamilton_96_tiprack_300uL_NTR
- Shelves: one per side of the U, per tier

In [ ]:
import datetime
import logging

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

simulation = True  # a facility this size exists only in simulation

protocol_mode = "simulation" if simulation else "execution"

run_identifier = "d1_s1_hamilton_facility_3d"

## The layout

Three tiers, each a U: a back row facing the open side, and two arms turned to face inwards.

In [ ]:
from pylabrobot.hamilton.prep import Prep
from pylabrobot.hamilton.star.device import STAR, STARlet, STARPlus

LEVELS = 3
LEVEL_HEIGHT = 1100.0  # floor to floor, mm
SHELF_THICKNESS = 40.0
GAP = 150.0  # between neighbours on a shelf, mm
ARM_DEPTH = 850.0  # how deep the side shelves are, front to back, mm
MARGIN = 300.0

# Left to right along the back, and front to back along each arm.
BACK = ["STARlet", "Prep", "STAR", "Prep", "STARlet"]
LEFT = ["STARplus", "Prep"]
RIGHT = ["Prep", "STAR"]
FACTORIES = {"STAR": STAR, "STARlet": STARlet, "STARplus": STARPlus, "Prep": Prep}

## One device, stocked

Every device is built simulated and gets labware, with liquid in its plates so the wells show it.

In [ ]:
from pylabrobot.resources import set_volume_tracking
from pylabrobot.resources.corning import cor_96_wellplate_360uL_Fb
from pylabrobot.resources.hamilton import (
    hamilton_plate_carrier_L5_ac,
    hamilton_tip_carrier_L5,
    hamilton_96_tiprack_300uL_NTR,
    hamilton_96_tiprack_1000uL,
    hamilton_96_tiprack_1000uL_filter,
)
from pylabrobot.resources.resource import Resource

set_volume_tracking(True)

# A STAR deck's `size_z` is its working envelope; the X-arm's top is the height it is drawn to.
STAR_DECK_HEIGHT = 334.7 + 140.0


def make_device(kind: str, name: str) -> Resource:
    device = FACTORIES[kind](simulation=simulation, name=name)
    if kind == "Prep":
        device.deck[0] = plate = cor_96_wellplate_360uL_Fb(name=f"{name}_plate")
        device.deck[1] = hamilton_96_tiprack_300uL_NTR(name=f"{name}_tips", with_tips=True)
        for i, well in enumerate(plate.get_all_items()):
            well.tracker.set_volume(40.0 + 3.0 * (i % 12) * 8)
        return device
    tips = hamilton_tip_carrier_L5(name=f"{name}_tip_carrier")
    tips[0] = hamilton_96_tiprack_1000uL(name=f"{name}_tips_0")
    tips[1] = hamilton_96_tiprack_1000uL_filter(name=f"{name}_tips_1")
    device.deck.assign_child_resource(tips, track=1)
    for carrier_index, track in enumerate((7, 13)):
        carrier = hamilton_plate_carrier_L5_ac(name=f"{name}_plates_{carrier_index}")
        for slot in range(5):
            carrier[slot] = plate = cor_96_wellplate_360uL_Fb(name=f"{name}_p{carrier_index}{slot}")
            for i, well in enumerate(plate.get_all_items()):
                well.tracker.set_volume((slot * 70.0 + 5.0 * (i % 8) + 20.0) % 350.0)
        device.deck.assign_child_resource(carrier, track=track)
    device.deck._size_z = STAR_DECK_HEIGHT
    device.deck._local_size_z = STAR_DECK_HEIGHT
    return device

## The facility

A `Facility` is a plain resource: shelves are its children, and the devices stand on the shelves.
The arms are rotated about Z, so their fronts face into the U.

In [ ]:
from collections import Counter

from pylabrobot.resources.coordinate import Coordinate
from pylabrobot.resources.rotation import Rotation
from pylabrobot.visualizer3D import Facility


def row_length(devices) -> float:
    return sum(d.get_size_x() for d in devices) + GAP * (len(devices) - 1)


def build() -> Facility:
    levels = []
    for level in range(LEVELS):
        tag = f"L{level + 1}"
        levels.append(
            {
                side: [make_device(kind, f"{tag}_{side}_{i}_{kind}") for i, kind in enumerate(kinds)]
                for side, kinds in (("back", BACK), ("left", LEFT), ("right", RIGHT))
            }
        )

    back_width = max(row_length(lv["back"]) for lv in levels)
    arm_length = max(row_length(lv[s]) for lv in levels for s in ("left", "right"))
    back_depth = max(d.get_size_y() for lv in levels for d in lv["back"]) + 100.0
    facility = Facility(
        name="facility",
        size_x=back_width + 2 * MARGIN,
        size_y=arm_length + GAP + back_depth + 2 * MARGIN,
        size_z=LEVELS * LEVEL_HEIGHT + 400.0,
    )

    back_y = MARGIN + arm_length + GAP  # the back row's front edge
    for level, lv in enumerate(levels):
        z = level * LEVEL_HEIGHT
        tag = f"L{level + 1}"
        shelves = {
            "back": Resource(f"{tag}_shelf_back", back_width, back_depth, SHELF_THICKNESS, category="bench"),
            "left": Resource(f"{tag}_shelf_left", ARM_DEPTH, arm_length, SHELF_THICKNESS, category="bench"),
            "right": Resource(f"{tag}_shelf_right", ARM_DEPTH, arm_length, SHELF_THICKNESS, category="bench"),
        }
        facility.assign_child_resource(shelves["back"], location=Coordinate(MARGIN, back_y, z))
        facility.assign_child_resource(shelves["left"], location=Coordinate(MARGIN, MARGIN, z))
        facility.assign_child_resource(
            shelves["right"], location=Coordinate(MARGIN + back_width - ARM_DEPTH, MARGIN, z)
        )

        # Back row: facing the open side of the U, as built.
        x = (back_width - row_length(lv["back"])) / 2
        for device in lv["back"]:
            shelves["back"].assign_child_resource(device, location=Coordinate(x, 0, SHELF_THICKNESS))
            x += device.get_size_x() + GAP

        # Left arm: turned +90 about Z, its front towards +x.
        y = 0.0
        for device in lv["left"]:
            device.rotation = Rotation(z=90)
            shelves["left"].assign_child_resource(device, location=Coordinate(ARM_DEPTH, y, SHELF_THICKNESS))
            y += device.get_size_x() + GAP

        # Right arm: turned -90, its front towards -x.
        y = arm_length
        for device in lv["right"]:
            device.rotation = Rotation(z=-90)
            shelves["right"].assign_child_resource(device, location=Coordinate(0, y, SHELF_THICKNESS))
            y -= device.get_size_x() + GAP
    return facility


facility = build()

devices = [r for r in facility.get_all_children() if r.category == "device"]
print(f"{len(devices)} devices, {len(facility.get_all_children()):,} resources")
print(dict(Counter(d.model for d in devices)))

## Setting the devices up

`setup()` builds what moves: the X-arm, the pipetting channels, and on a STAR the 96-head, the
iSWAP and the autoload. Before it, a device is only its body and its deck.

In [ ]:
import asyncio

await asyncio.gather(*(device.setup() for device in devices))

moving = ("x_arm", "pipette_channel", "head96", "iswap_head", "autoload_sled")
print(dict(Counter(r.category for r in facility.get_all_children() if r.category in moving)))

## Watching it in 3D

The viewer serves the tree to a browser and keeps it current. Open the link it prints; it runs for
as long as this kernel does.

In [ ]:
import webbrowser

from pylabrobot.visualizer3D import Viewer3D

viewer = Viewer3D(facility, name="D1-S1: Hamilton facility", open_browser=False)
await viewer.start()

url = viewer.url.replace("/#", "/?view=iso#")  # start in the ISO view
print(url)
webbrowser.open(url)

await viewer.wait_for_browser(timeout=120)
print("a browser is drawing the facility")

## Synchronous Movement

In [ ]:
import asyncio
from tqdm.auto import tqdm

for x in tqdm(range(6)):
  await asyncio.gather(
    *(device.x_arm.move_to_x_position(device.deck.get_size_x() / 6) for device in devices)
  )
  await asyncio.sleep(3)

  await asyncio.gather(
    *(device.x_arm.move_to_x_position(device.deck.get_size_x() / 1.5) for device in devices)
  )
  await asyncio.sleep(3)

## Asynchronous Movement

In [ ]:
async def shuttle(device, n_cycles: int, pause: float, position: int):
  size_x = device.deck.get_size_x()
  for _ in tqdm(range(n_cycles), desc=f"device {position}", position=position):
    await device.x_arm.move_to_x_position(size_x / 6)
    await asyncio.sleep(pause)
    await device.x_arm.move_to_x_position(size_x / 1.5)
    await asyncio.sleep(pause)

In [ ]:

await asyncio.gather(
  *(shuttle(device, n_cycles=5, pause=2 + i, position=i) for i, device in enumerate(devices))
)

# STOP

## Stop the viewer and the devices

In [ ]:
await viewer.stop()
for device in devices:
    await device.stop()